# 데이터 점검

빌드된 instruction 데이터를 눈으로 확인하는 노트북입니다. 학습 전에

- 태스크별로 몇 개가 있는지
- 프롬프트와 정답이 실제로 어떻게 생겼는지
- CoT의 rationale이 answer를 실제로 뒷받침하는지
- `det_objects`가 받는 1프레임 비전 + 1초 레이더 창이 의도대로인지

를 확인합니다. GPU가 필요 없습니다.

In [1]:
import os, sys, json, re, textwrap

sys.path.insert(0, os.path.dirname(os.getcwd()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from datatools import paths

COMMON = paths.COMMON_DIR
ITEMS = os.path.join(COMMON, "instruct_items_tasks01_06.parquet")
pd.set_option("display.width", 170)
pd.set_option("display.max_colwidth", 90)

print("split root :", paths.SPLIT_ROOT)
print("common     :", COMMON)

split root : /NHNHOME/workspace/dataset/raw_Auto_datasets/preprocessed_train_test_split
common     : /NHNHOME/workspace/dataset/raw_Auto_datasets/preprocessed_train_test_split/common


## 1. 어떤 파일이 있고 얼마나 큰가

In [2]:
files = ["instruct_items_tasks01_06.parquet", "scene_features_all_clips.parquet",
         "radar_object_probes.parquet", "radar_structure_probes.parquet",
         "nvidia_clips.parquet", "qa_holdout_clips.json"]
rows = []
for f in files:
    p = os.path.join(COMMON, f)
    rows.append({"file": f, "exists": os.path.exists(p),
                 "MB": round(os.path.getsize(p) / 1e6, 1) if os.path.exists(p) else None})
pd.DataFrame(rows)

,file,exists,MB
0,instruct_items_tasks01_06.parquet,True,481.4
1,scene_features_all_clips.parquet,True,326.9
2,radar_object_probes.parquet,True,14.8
3,radar_structure_probes.parquet,True,7.6
4,nvidia_clips.parquet,True,48.3
5,qa_holdout_clips.json,True,0.0


## 2. 태스크별 아이템 수

태스크 01~06과 그 CoT 변형이 한 파일에 들어 있고 `task` 컬럼으로 나뉩니다.

In [3]:
items = pd.read_parquet(ITEMS, columns=["clip_id", "task", "frame", "split"])
print(f"{len(items):,} rows, {items.clip_id.nunique():,} clips")

(items.pivot_table(index="task", columns="split", values="clip_id",
                   aggfunc="count", fill_value=0)
      .assign(total=lambda d: d.sum(axis=1))
      .sort_values("total", ascending=False))

7,171,214 rows, 173,405 clips


split,test,train,val,total
task,,,,
det_objects,216912,506958,316560,1040430
det_objects_cot,170565,433590,260931,865086
plan_ego,108456,253479,158280,520215
track_identity,108456,253479,158280,520215
plan_ego_cot,108456,253479,158280,520215
world_model,108399,253374,158202,519975
track_identity_cot,98298,231764,144044,474106
world_model_cot,98298,231764,144044,474106
motion_seg,92787,235695,142089,470571


## 3. 앵커 프레임

각 태스크가 클립의 어느 시점에서 만들어졌는지입니다. `frame`은 1-indexed이고 t = frame − 1초.

- `det_objects`는 **3/6/9/12/15/18초** — 미래를 묻지 않으므로 3초 지평선 제약이 없습니다
- 나머지는 **5/10/15초** — 3초 예측이 클립 안에 들어가야 합니다

In [4]:
anchors = (items.groupby("task")["frame"].agg(lambda s: sorted(s.unique()))
                .to_frame("frames"))
anchors["seconds"] = anchors.frames.map(lambda fs: [f - 1 for f in fs])
anchors["per_clip"] = (items.groupby("task").size()
                       / items.groupby("task")["clip_id"].nunique()).round(2)
anchors

,frames,seconds,per_clip
task,,,
agent_traj,"[6, 11, 16]","[5, 10, 15]",2.15
agent_traj_cot,"[6, 11, 16]","[5, 10, 15]",1.87
depth_range,"[6, 11, 16]","[5, 10, 15]",3.00
depth_range_cot,"[6, 11, 16]","[5, 10, 15]",2.76
det_objects,"[4, 7, 10, 13, 16, 19]","[3, 6, 9, 12, 15, 18]",6.00
det_objects_cot,"[4, 7, 10, 13, 16, 19]","[3, 6, 9, 12, 15, 18]",5.64
motion_seg,"[6, 11, 16]","[5, 10, 15]",3.00
motion_seg_cot,"[6, 11, 16]","[5, 10, 15]",2.76
plan_ego,"[6, 11, 16]","[5, 10, 15]",3.00


## 4. 프롬프트와 정답 살펴보기

`TASK`를 바꿔 가며 확인하세요.

In [5]:
TASK = "det_objects"
N = 3

full = pd.read_parquet(ITEMS)
for _, r in full[full.task == TASK].head(N).iterrows():
    print("=" * 100)
    print(f"clip {r.clip_id}   frame {r.frame} (t={r.frame - 1}s)   split {r.split}")
    print("PROMPT :", textwrap.fill(r.prompt, 96, subsequent_indent=" " * 9))
    print("TARGET :", textwrap.fill(r.target, 96, subsequent_indent=" " * 9))

clip 25cd4769-5dcf-4b53-a351-bf2c5deb6124   frame 4 (t=3s)   split train
PROMPT : List every road user in the forward sector with its class, range and azimuth.
TARGET : automobile 8 m az +59 deg stationary; automobile 12 m az +35 deg stationary; automobile 18 m az
         +22 deg stationary; automobile 25 m az +16 deg stationary; automobile 33 m az +12 deg
         stationary; automobile 38 m az +6 deg moving; automobile 58 m az +3 deg moving;
         automobile 60 m az -0 deg moving
clip 25cd4769-5dcf-4b53-a351-bf2c5deb6124   frame 7 (t=6s)   split train
PROMPT : List every road user in the forward sector with its class, range and azimuth.
TARGET : automobile 17 m az +12 deg moving; automobile 28 m az +6 deg moving; automobile 37 m az -30 deg
         stationary; automobile 45 m az +4 deg moving; automobile 46 m az -0 deg moving
clip 25cd4769-5dcf-4b53-a351-bf2c5deb6124   frame 10 (t=9s)   split train
PROMPT : List every road user in the forward sector with its class, range and azim

## 5. CoT: rationale이 answer를 뒷받침하는가

`_cot` 태스크는 `{"rationale": ..., "answer": ...}` 형식입니다. rationale은 감상이 아니라
라벨에서 계산된 증거여야 하고, 그것이 answer를 결정해야 합니다. 근거를 따라갔을 때
답이 나오지 않으면 그 사슬은 잘못된 것이고, 보상을 걸면 모델이 그 잘못된 사슬을 배웁니다.

In [6]:
cot = sorted(t for t in full.task.unique() if t.endswith("_cot"))
print("CoT 태스크:", cot)

for task in cot:
    r = full[full.task == task].iloc[0]
    try:
        d = json.loads(r.target)
    except json.JSONDecodeError:
        print(f"{task}: JSON 아님")
        continue
    print("\n" + "=" * 100)
    print(f"### {task}")
    print("Q :", textwrap.fill(r.prompt, 96, subsequent_indent=" " * 4))
    print("R :", textwrap.fill(d["rationale"], 96, subsequent_indent=" " * 4))
    print("A :", textwrap.fill(d["answer"], 96, subsequent_indent=" " * 4))

CoT 태스크: ['agent_traj_cot', 'depth_range_cot', 'det_objects_cot', 'motion_seg_cot', 'plan_ego_cot', 'track_identity_cot', 'world_model_cot']



### agent_traj_cot
Q : Track #8 is a automobile at 25 m, azimuth +8 deg. Where will it be over the next 3 seconds?
R : The radar puts 4 returns on track #8 at 25 m, median radial velocity -7.4 m/s, so it is closing.
A : +1s 17 m az +12 deg; +2s 9 m az +24 deg



### depth_range_cot
Q : How far is the nearest object ahead, and the nearest one the radar confirms?
R : automobile at 12 m: 6 returns; automobile at 25 m: 4 returns; automobile at 62 m: 1 returns.
A : nearest automobile at 12 m; nearest radar-confirmed automobile at 12 m



### det_objects_cot
Q : List every road user in the forward sector with its class, range and azimuth.
R : radar-confirmed: automobile at 12 m: 6 radar returns, 6 of them moving once the ego's own motion
    is removed; automobile at 18 m: 5 radar returns, 0 of them moving once the ego's own motion
    is removed; automobile at 25 m: 4 radar returns, 0 of them moving once the ego's own motion
    is removed; automobile at 33 m: 2 radar returns, 0 of them moving once the ego's own motion
    is removed; automobile at 38 m: 5 radar returns, 5 of them moving once the ego's own motion
    is removed; automobile at 58 m: 1 radar returns, 1 of them moving once the ego's own motion
    is removed. camera only: automobile at 8 m; automobile at 60 m.
A : automobile 8 m az +59 deg stationary; automobile 12 m az +35 deg stationary; automobile 18 m az
    +22 deg stationary; automobile 25 m az +16 deg stationary; automobile 33 m az +12 deg
    stationary; automobile 38 m az +6 deg moving; automobi


### motion_seg_cot
Q : Which objects ahead are moving and which are stationary? Use the radar Doppler.
R : automobile at 12 m: 6 returns, 6 of them still moving once the ego's own motion is removed;
    automobile at 25 m: 4 returns, 4 of them still moving once the ego's own motion is removed;
    automobile at 62 m: 1 returns, 0 of them still moving once the ego's own motion is removed.
    An object whose returns keep a residual above 1 m/s is moving; one whose returns are
    explained entirely by the ego's own motion is not.
A : moving: automobile 25 m az +8 deg (4 radar returns), automobile 38 m az +4 deg (no radar
    return), automobile 50 m az -0 deg (no radar return), automobile 56 m az +2 deg (no radar
    return), automobile 66 m az -2 deg (no radar return). stationary: automobile 12 m az +37 deg
    (6 radar returns), automobile 46 m az -24 deg (no radar return), automobile 62 m az -28 deg
    (1 radar return).



### plan_ego_cot
Q : Predict the ego vehicle's path over the next 3 seconds as (x, y) offsets in metres.
R : The ego vehicle is travelling at 11.1 m/s and will brake and go straight. At that speed it
    covers about 11 m per second.
A : +1s (+10.9, -0.0); +2s (+21.2, -0.1); +3s (+31.0, -0.1)



### track_identity_cot
Q : Give the track id, class, range and age of every object you are tracking ahead.
R : #130 first seen at t=0.2 s, now t=5 s; #8 first seen at t=0.0 s, now t=5 s; #3 first seen at
    t=0.0 s, now t=5 s; #158 first seen at t=4.6 s, now t=5 s; #2 first seen at t=0.0 s, now t=5
    s; #21 first seen at t=0.0 s, now t=5 s; #160 first seen at t=4.7 s, now t=5 s; #159 first
    seen at t=4.7 s, now t=5 s. Age is the difference between the two.
A : #130 automobile 12 m visible 4.8 s; #8 automobile 25 m visible 5.0 s; #3 automobile 38 m visible
    5.0 s; #158 automobile 46 m visible 0.4 s; #2 automobile 50 m visible 5.0 s; #21 automobile
    56 m visible 5.0 s; #160 automobile 62 m visible 0.3 s; #159 automobile 66 m visible 0.3 s



### world_model_cot
Q : The ego vehicle will brake and go straight over the next 3 seconds. What will the forward scene
    look like then?
R : Now: automobile at 12 m, az +37 deg, radial -7.4 m/s; automobile at 25 m, az +8 deg, radial -7.4
    m/s; automobile at 38 m, az +4 deg; automobile at 46 m, az -24 deg; automobile at 50 m, az
    -0 deg; automobile at 56 m, az +2 deg; automobile at 62 m, az -28 deg, radial -9.9 m/s;
    automobile at 66 m, az -2 deg. The ego covers about 33 m in 3 s, so ranges change by that
    much plus each object's own motion, and anything the ego draws level with leaves the forward
    sector.
A : 7 automobiles, 1 person; 2 moving; nearest automobile at 8 m


## 6. det_objects의 입력

`det_objects`만 입력이 다릅니다.

| | 일반 태스크 | det_objects |
|---|---|---|
| 비전 | 20프레임 (1 Hz × 20 s) | **1프레임** (질문한 순간) |
| 레이더 | 20스캔 (1 Hz × 20 s) | **20스캔** (질문 직전 약 1 s, 센서 원래 속도) |

레이더 원본은 LRR·MRR이 20 Hz라 20초 클립에 약 400스캔이 있는데, 기본 모드는 초당
1개만 남기고 95%를 버립니다. 한 순간을 묻는 질문에는 그 반대가 맞습니다.

In [7]:
from datatools.frame_objects import read_member

clips = pd.read_parquet(os.path.join(COMMON, "nvidia_clips.parquet"))
lrr = clips[clips.has_lrr1.fillna(False) & clips.has_radar_extrinsics.fillna(False)]
row = lrr.loc[lrr.index[0]]

radar = read_member(paths.NVIDIA_ROOT, row.radar_lrr1_zip, row.radar_lrr1_member)
scans = np.sort(radar.timestamp.unique()) / 1e6
period = float(np.median(np.diff(scans)))
print(f"클립 전체 스캔 {len(scans)}개, 주기 {period:.3f} s ({1 / period:.1f} Hz)")
print()

for until in (3, 6, 9, 12, 15, 18):
    w = scans[scans <= until][-20:]
    print(f"  t={until:>2}s 창 : {len(w):>2}개  {w.min():6.3f}..{w.max():6.3f}"
          f"  span {w.max() - w.min():.3f} s")

default = np.array([scans[np.argmin(np.abs(scans - f))] for f in range(20)])
print(f"\n  기본 모드  : {len(default)}개  {default.min():6.3f}..{default.max():6.3f}"
      f"  span {default.max() - default.min():.3f} s")

클립 전체 스캔 399개, 주기 0.050 s (20.0 Hz)

  t= 3s 창 : 20개   2.011.. 2.961  span 0.950 s
  t= 6s 창 : 20개   5.006.. 5.961  span 0.955 s
  t= 9s 창 : 20개   8.006.. 8.956  span 0.950 s
  t=12s 창 : 20개  11.002..11.956  span 0.954 s
  t=15s 창 : 20개  14.006..14.956  span 0.950 s
  t=18s 창 : 20개  17.006..17.957  span 0.950 s

  기본 모드  : 20개   0.056..19.007  span 18.952 s


## 7. 레이더 스캔 한 장 (BEV)

자차가 원점, 위쪽이 전방입니다. 색은 도플러 잔차 — 자차 운동을 제거하고도 남은 속도이고,
1 m/s를 넘으면 실제로 움직이는 것으로 판정합니다. 빨간 점선이 라벨이 사용하는 ±60° 섹터입니다.

In [8]:
from datatools.frame_objects import ego_frame, radar_scan, SENSOR_NAME

clip_id = lrr.index[0]
ego = read_member(paths.NVIDIA_ROOT, row.egomotion_zip, row.egomotion_member)
derived = ego_frame(ego)
ext = pd.read_parquet(os.path.join(paths.NVIDIA_ROOT, row.radar_extrinsics_parquet))
scan = radar_scan(radar, ext.loc[clip_id], SENSOR_NAME["lrr1"], 12.0, ego, derived)

rig, residual, moving = scan["rig"], scan["residual"], scan["moving"]
fig, ax = plt.subplots(figsize=(7, 7))
sc = ax.scatter(rig[:, 1], rig[:, 0], c=np.abs(residual), s=6, cmap="viridis",
                vmin=0, vmax=8)
ax.scatter(0, 0, marker="^", s=180, c="red", label="ego")
for deg in (-60, 60):
    a = np.radians(deg)
    ax.plot([0, 200 * np.sin(a)], [0, 200 * np.cos(a)], "r--", lw=0.8, alpha=0.5)
ax.set_xlabel("left (m)")
ax.set_ylabel("forward (m)")
ax.set_title(f"imaging LRR @ t=12 s - {len(rig)} returns, {int(moving.sum())} moving")
ax.set_xlim(-120, 120)
ax.set_ylim(-10, 200)
ax.set_aspect("equal")
ax.grid(alpha=0.25)
ax.legend(loc="upper right")
plt.colorbar(sc, ax=ax, label="|Doppler residual| (m/s)")
plt.tight_layout()
plt.show()

## 8. 비디오 프레임

`det_objects`가 실제로 받는 한 장과, 일반 태스크가 받는 20장의 일부입니다.

In [9]:
from training.video_frames import clip_frames, pad_frames

frames = pad_frames(clip_frames(paths.NVIDIA_ROOT, row))
SECONDS = 12

fig, axes = plt.subplots(2, 1, figsize=(13, 6))
axes[0].imshow(frames[SECONDS])
axes[0].set_title(f"det_objects 입력 - t={SECONDS}s, 한 장")
axes[1].imshow(np.concatenate([np.asarray(f) for f in frames[:6]], axis=1))
axes[1].set_title("일반 태스크 입력 - 20장 중 앞 6장 (1 Hz)")
for a in axes:
    a.axis("off")
plt.tight_layout()
plt.show()

/tmp/ipykernel_1934875/1863109246.py:13: UserWarning: Glyph 51077 (\N{HANGUL SYLLABLE IB}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_1934875/1863109246.py:13: UserWarning: Glyph 47141 (\N{HANGUL SYLLABLE RYEOG}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_1934875/1863109246.py:13: UserWarning: Glyph 54620 (\N{HANGUL SYLLABLE HAN}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_1934875/1863109246.py:13: UserWarning: Glyph 51109 (\N{HANGUL SYLLABLE JANG}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_1934875/1863109246.py:13: UserWarning: Glyph 51068 (\N{HANGUL SYLLABLE IL}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_1934875/1863109246.py:13: UserWarning: Glyph 48152 (\N{HANGUL SYLLABLE BAN}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_1934875/1863109246.py:13: UserWarning: Glyph 53468 (\N{HANGUL SYLLABLE TAE}) missing from font(s) Deja

## 9. 보상 함수를 실제 아이템에 적용

RLVR 보상은 평가 채점기에서 유도했습니다. 정답을 그대로 넣으면 1.0이 나와야 하고,
숫자를 망가뜨리면 떨어져야 합니다. 새 태스크를 추가했다면 여기서 먼저 확인하세요.

In [10]:
from training.task_scorers import reward_for

def double_numbers(text):
    """형식은 그대로 두고 숫자만 2배로 -- 내용만 틀린 답을 만든다."""
    return re.sub(r"\d+(?:\.\d+)?",
                  lambda m: str(round(float(m.group()) * 2, 1)), text)

rows = []
for task in sorted(full.task.unique()):
    fn = reward_for(task)
    r = full[full.task == task].iloc[0]
    rows.append({"task": task,
                 "reward": fn.__name__ if fn else "없음 (검증 불가)",
                 "정답": round(fn(r.target, r.target), 3) if fn else None,
                 "숫자 2배": round(fn(double_numbers(r.target), r.target), 3) if fn else None})
pd.DataFrame(rows)

,task,reward,정답,숫자 2배
0,agent_traj,reward_trajectory,1.00,0.000
1,agent_traj_cot,reward_with_rationale[agent_traj],1.00,0.150
2,depth_range,reward_quantity,1.00,0.000
3,depth_range_cot,reward_with_rationale[depth_range],0.85,0.000
4,det_objects,reward_objects,1.00,0.000
5,det_objects_cot,reward_with_rationale[det_objects],1.00,0.172
6,motion_seg,reward_objects,1.00,0.000
7,motion_seg_cot,reward_with_rationale[motion_seg],1.00,0.165
8,plan_ego,reward_waypoints,1.00,0.000
9,plan_ego_cot,reward_with_rationale[plan_ego],0.85,0.000


## 10. 레이더 프로브의 오염도

평가 전용 프로브입니다. **오염도**는 프로브 정답이 인코더가 지도학습한 스칼라
(`n_points`, `n_moving`, `max_rcs`)와 갖는 최대 상관입니다. 이 값이 크면 모델이 그
스칼라만 읽어도 프로브를 풀 수 있으므로, 그 프로브로는 레이더 이해를 잴 수 없습니다.
`radar_probe`가 그렇게 순환 측정이 됐습니다.

In [11]:
NUM = re.compile(r"-?\d+(?:\.\d+)?")
feat = pd.read_parquet(os.path.join(COMMON, "scene_features_all_clips.parquet"),
                       columns=["clip_id", "frame", "lrr1_n_points",
                                "lrr1_n_moving", "lrr1_max_rcs"])
SUPERVISED = ["lrr1_n_points", "lrr1_n_moving", "lrr1_max_rcs"]

out = []
for name in ("radar_structure_probes.parquet", "radar_object_probes.parquet"):
    path = os.path.join(COMMON, name)
    if not os.path.exists(path):
        continue
    pr = pd.read_parquet(path)
    m = (pr[pr.split == "test"].merge(feat, on=["clip_id", "frame"])
                               .dropna(subset=["lrr1_n_points"]))
    m = m.assign(first=m.target.map(lambda s: float(NUM.findall(s)[0])))
    for form, g in m.groupby("form"):
        worst = max(abs(float(np.corrcoef(g["first"], g[c])[0, 1]))
                    for c in SUPERVISED)
        out.append({"probe": name.split("_")[1], "form": form, "n": len(g),
                    "오염도": round(worst, 3),
                    "판정": "깨끗" if worst < 0.3 else "부분적" if worst < 0.7 else "오염"})
pd.DataFrame(out).sort_values("오염도")

,probe,form,n,오염도,판정
5,object,obj_azimuth,6428,0.032,깨끗
0,structure,bearing,8523,0.035,깨끗
6,object,obj_closing,6400,0.082,깨끗
2,structure,lateral,8469,0.123,깨끗
7,object,obj_gap,5233,0.156,깨끗
9,object,obj_range,6514,0.249,깨끗
8,object,obj_moving,7251,0.423,부분적
4,structure,spread,8202,0.458,부분적
1,structure,closing,8289,0.562,부분적
3,structure,near_far,8292,0.690,부분적
